<a href="https://colab.research.google.com/github/angelagdca/GDP_first_model/blob/main/GDP_model_firstry.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **PIB España (BISTRO)**

###**Step 1**

In [ ]:
#Deja listo el entorno para usar BISTRO/Moirai

import os
import sys
import subprocess
from pathlib import Path

#Compatibilidad de NumPy (sobre todo en Google Colab)
try:
    import numpy as np

    if np.__version__.startswith("2."):
        print(f"Current NumPy is {np.__version__}. Downgrading to 1.26.4...")

        subprocess.run([
            "pip", "install", "-q", "--force-reinstall",
            "numpy==1.26.4",
            "pandas==2.1.4",
            "scipy==1.11.4",
            "opencv-python==4.8.0.76"
        ], check=True)

        print("✅ Install complete. Restarting runtime automatically...")
        os.kill(os.getpid(), 9)

    else:
        print(f"✅ Success! Using NumPy version: {np.__version__}")

except Exception as e:
    print(f"NumPy compatibility check skipped/error: {e}")

#Si estás en Google Colab: clonar repo e instalar deps

if 'google.colab' in sys.modules:
    print("Running in Google Colab. Cloning BISTRO repo...")

    if not os.path.exists("/content/bistro"):
        !git clone -q https://github.com/bis-med-it/bistro.git

    print("Installing project dependencies...")
    !pip install -q -r /content/bistro/requirements.txt

    print("✅ Repository and dependencies ready.")

✅ Success! Using NumPy version: 1.26.4
Running in Google Colab. Cloning BISTRO repo...
Installing project dependencies...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.2/114.2 kB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.2/147.2 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.8/153.8 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.5/153.5 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.7/536.7 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 81.4 MB/s eta 0:00:00


In [ ]:
#Ajustar rutas del proyecto

if 'google.colab' in sys.modules:
    project_root = Path("/content/bistro")
else:
    # Cambia esta ruta si trabajas en local
    project_root = Path.cwd()

repo_root = project_root
src_root = repo_root / "src"
script_root = repo_root / "script"

if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

if str(script_root) not in sys.path:
    sys.path.insert(0, str(script_root))

print("repo_root:", repo_root)
print("src_root:", src_root)
print("script_root:", script_root)

#Imports principales
import numpy as np
import pandas as pd

from gluonts.dataset.pandas import PandasDataset
from gluonts.dataset.split import split

from uni2ts.model.moirai import MoiraiForecast, MoiraiModule

from inference_util import plot_publication_forecast_comparison
from preprocessing_util import (
    aggregate_daily_forecast_to_monthly,
    prepare_yoy_monthly_for_daily_inference,
)

print("✅ Core imports loaded successfully.")

repo_root: /content/bistro
src_root: /content/bistro/src
script_root: /content/bistro/script
✅ Core imports loaded successfully.


###**Step 2**

In [ ]:
from pathlib import Path

data_dir = Path("/content/bistro/data")
data_dir.mkdir(parents=True, exist_ok=True)

print("Carpeta creada en:", data_dir)

Carpeta creada en: /content/bistro/data
